In [1]:
print("Test")

Test


In [2]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [47]:
import json
import estnltk
import tqdm
import pandas as pd
from estnltk.default_resolver import make_resolver

from scripts.model import initialize_model
from scripts.model.bert_morph_tagger import BertMorphTagger

In [48]:
conflicts_dir = DATA_DIR / "conflicts"
bmt = BertMorphTagger(model_location=str(MODEL_DIR / "NER_mudel_v2"))

### Adding Bert Morph v2 predictions to the morph-syntax conflicts dataset


In [75]:
# Open JSON file
with open(
    conflicts_dir / "raw" / "nsubj_conflicts_dataset_15022026.json",
    "r",
    encoding="utf-8",
) as f:
    # Load JSON data
    conflicts_data = json.load(f)

In [76]:
# Convert the updated data to a DataFrame for easier analysis
conflicts_df = pd.DataFrame(conflicts_data)
display(conflicts_df.head())

# Convert list values (e.g., spans, forms) to tuples so they are hashable
list_like_cols = [
    col
    for col in conflicts_df.columns
    if conflicts_df[col].apply(lambda x: isinstance(x, list)).any()
]
for col in list_like_cols:
    conflicts_df[col] = conflicts_df[col].apply(
        lambda x: tuple(x) if isinstance(x, list) else x
    )

# Check duplicates based on the row content
duplicates = conflicts_df.duplicated(keep=False)
duplicate_rows = conflicts_df[duplicates]

# Sort duplicate rows for better readability
duplicate_rows = duplicate_rows.sort_values(by=conflicts_df.columns.tolist())
print(f"Number of duplicate rows: {duplicate_rows.shape[0]}")
print("Duplicate rows:")
display(duplicate_rows)

# Remove duplicates
conflicts_df_no_duplicates = conflicts_df.drop_duplicates()
print(f"Number of rows before removing duplicates: {conflicts_df.shape[0]}")
print(f"Number of rows after removing duplicates: {conflicts_df_no_duplicates.shape[0]}")

,sentence_id,sentence,word_form,span,current_deprel,current_case,current_analysis
0,12410790,"Kristina Shmiguni ahistas Holmenkollenis 30 km klassikatehnikasõidu lõpukilomeetritel kurnatus , ent viiendast kohast maailma karikasarja liidrirüü hoidmiseks piisas .",Kristina,"[0, 8]",nsubj,gen,"gen,prop,sg"
1,16715193,"Küpsetamist-kaunistamist jätkub tegelikult aasta lõpuni : sel pühapäeval aitab Torupilli Selveris piparkooke teha ja valmis piparkoogimaja detaile kokku panna pagar , järgmisel pühapäeval on tal kaubamaja kodumaailmas abiks Pipi .",Torupilli,"[79, 88]",nsubj,gen,"gen,prop,sg"
2,214160,"Kui 20. sajandi algul laenu- ja hoiuühistud aitasid vältida eestlaste hoiustevoolu Venemaale , siis 21. sajandi algul võiks neist abi olla hoiuste äravoolu takistamisel ääremaalt ettevõtluskeskustesse .",laenu-,"[22, 28]",nsubj,gen,"com,gen,sg"
3,225510,Tarkvara aitab jagu saada mitmes ametkonnas võidutsevast liigsest bürokraatiast .,Tarkvara,"[0, 8]",nsubj,gen,"com,gen,sg"
4,325158,"Praegu aitavad selle kala DNA-st saadud elektritaju- ja elektriimpulsiomadused ühendada tehiselundite sensoorsed ja motoorsed signaalid tehisnaha pinnaga , kust närvisiirded neid edasi toimetavad .",kala,"[21, 25]",nsubj,gen,"com,gen,sg"


Number of duplicate rows: 70
Duplicate rows:


,sentence_id,sentence,word_form,span,current_deprel,current_case,current_analysis
4173,539504,""" Shaq on monstrum , "" oskas Rocketsi Antoine Carr teha vastaste hiiglasele komplimendi .",Rocketsi,"(29, 37)",nsubj,gen,"gen,prop,sg"
4185,539504,""" Shaq on monstrum , "" oskas Rocketsi Antoine Carr teha vastaste hiiglasele komplimendi .",Rocketsi,"(29, 37)",nsubj,gen,"gen,prop,sg"
6186,1370921,"Eesti Energia nõukogu ja Eesti läbirääkimisdelegatsioon tahavad teha majanduslikult põhjendatud lepingut , aga seda on üliraske teha , kui poliitikud samas deklareerivad , et lepingut on vaja iga hinna eest , ütles Käo kommentaariks peaminister Mart Laari seisukohavõtule reedeses Päevalehes .",nõukogu,"(14, 21)",nsubj,gen,"com,gen,sg"
6237,1370921,"Eesti Energia nõukogu ja Eesti läbirääkimisdelegatsioon tahavad teha majanduslikult põhjendatud lepingut , aga seda on üliraske teha , kui poliitikud samas deklareerivad , et lepingut on vaja iga hinna eest , ütles Käo kommentaariks peaminister Mart Laari seisukohavõtule reedeses Päevalehes .",nõukogu,"(14, 21)",nsubj,gen,"com,gen,sg"
6187,2280024,"Kui Kristina Šmigun on tahtnud teha puhast sporti ja temasse on usutud ning järsku hakatakse kahtlema , siis võib see talle reetmisena tunduda .",Kristina,"(4, 12)",nsubj,gen,"gen,prop,sg"
...,...,...,...,...,...,...,...
6647,17234045,“ Südamete murdumise maja ” olen tahtnud teha juba mitu aastat .,maja,"(21, 25)",nsubj,gen,"com,gen,sg"
5311,18670148,"Seaduseelnõu püüab teha ametnikele ettekirjutusi kindlate tähtaegade suhtes , mille jooksul nad peavad läbi vaatama nendele seaduse järgi esitatud avalduse ja muud dokumendid ning andma need kindlas rütmis korrus kõrgemale - omavalitsus maavalitsusele , maavalitsus kohalikule kinnistule .",Seaduseelnõu,"(0, 12)",nsubj,gen,"com,gen,sg"
5323,18670148,"Seaduseelnõu püüab teha ametnikele ettekirjutusi kindlate tähtaegade suhtes , mille jooksul nad peavad läbi vaatama nendele seaduse järgi esitatud avalduse ja muud dokumendid ning andma need kindlas rütmis korrus kõrgemale - omavalitsus maavalitsusele , maavalitsus kohalikule kinnistule .",Seaduseelnõu,"(0, 12)",nsubj,gen,"com,gen,sg"
4179,20225323,Tarzan_sad: klge keegi ftp oskab midagi teha olex nats abi vaja,klge,"(12, 16)",nsubj,gen,"com,gen,sg"


Number of rows before removing duplicates: 9634
Number of rows after removing duplicates: 9599


In [77]:
resolver = make_resolver()

In [78]:
# Process each row in the conflicts dataset
conflicts_df_no_duplicates = conflicts_df_no_duplicates.copy()
for index, row in tqdm.tqdm(conflicts_df_no_duplicates.iterrows(), total=conflicts_df_no_duplicates.shape[0]):
    sentence = row["sentence"]
    word = row["word_form"]
    word_span = row["span"]
    # print(f"Sentence: {sentence}")
    # print(f"Word: {word}, Span: {word_span}")
    text_obj = estnltk.Text(sentence)
    text_obj.tag_layer(["sentences", "words"])
    bmt.tag(text_obj)
    forms = []
    for ma in text_obj.bert_morph_tagging:
        if ma.start == word_span[0] and ma.end == word_span[1]:
            # Extract the form and pos
            for form in ma.form:
                forms.append(form)
    # Add a new column to the DataFrame with the extracted forms
    conflicts_df_no_duplicates.loc[index, "bert_morph_v2_form"] = forms

 78%|███████▊  | 7473/9599 [05:49<01:33, 22.69it/s]E:\Git_projects/EstNLTK/EstNLTK_model_training\scripts\model\bert_tokens_to_words_rewriter.py:228: UserWarning: (!) No matching words span for bert token Span('', [{'bert_tokens': '▁»', 'form': '', 'partofspeech': 'Z', 'probability': 0.99999}]).
  warnings.warn(
100%|██████████| 9599/9599 [07:27<00:00, 21.46it/s]


In [79]:
conflicts_df_no_duplicates.head()

,sentence_id,sentence,word_form,span,current_deprel,current_case,current_analysis,bert_morph_v2_form
0,12410790,"Kristina Shmiguni ahistas Holmenkollenis 30 km klassikatehnikasõidu lõpukilomeetritel kurnatus , ent viiendast kohast maailma karikasarja liidrirüü hoidmiseks piisas .",Kristina,"(0, 8)",nsubj,gen,"gen,prop,sg",[sg n]
1,16715193,"Küpsetamist-kaunistamist jätkub tegelikult aasta lõpuni : sel pühapäeval aitab Torupilli Selveris piparkooke teha ja valmis piparkoogimaja detaile kokku panna pagar , järgmisel pühapäeval on tal kaubamaja kodumaailmas abiks Pipi .",Torupilli,"(79, 88)",nsubj,gen,"gen,prop,sg",[sg g]
2,214160,"Kui 20. sajandi algul laenu- ja hoiuühistud aitasid vältida eestlaste hoiustevoolu Venemaale , siis 21. sajandi algul võiks neist abi olla hoiuste äravoolu takistamisel ääremaalt ettevõtluskeskustesse .",laenu-,"(22, 28)",nsubj,gen,"com,gen,sg",[sg g]
3,225510,Tarkvara aitab jagu saada mitmes ametkonnas võidutsevast liigsest bürokraatiast .,Tarkvara,"(0, 8)",nsubj,gen,"com,gen,sg",[sg n]
4,325158,"Praegu aitavad selle kala DNA-st saadud elektritaju- ja elektriimpulsiomadused ühendada tehiselundite sensoorsed ja motoorsed signaalid tehisnaha pinnaga , kust närvisiirded neid edasi toimetavad .",kala,"(21, 25)",nsubj,gen,"com,gen,sg",[sg g]


In [81]:
# Save the updated data back to a new JSON file
save_path = (
    conflicts_dir
    / "processed"
    / "nsubj_conflicts_dataset_15022026_with_Bert_Morph_v2.json"
)
print(f"Saving updated data to {save_path}")
conflicts_df_no_duplicates.to_json(save_path, orient="records", force_ascii=False, indent=1)
print(f"Data saved successfully to {save_path}")

Saving updated data to ..\data\conflicts\processed\nsubj_conflicts_dataset_15022026_with_Bert_Morph_v2.json
Data saved successfully to ..\data\conflicts\processed\nsubj_conflicts_dataset_15022026_with_Bert_Morph_v2.json


### Inspecting the morph-syntax conflicts dataset with Bert Morph v2 predictions


In [129]:
# Open the updated JSON file to verify the changes

save_path = (
    conflicts_dir
    / "processed"
    / "nsubj_conflicts_dataset_15022026_with_Bert_Morph_v2.json"
)
# Open JSON file as a DataFrame
conflicts_df = pd.read_json(save_path, orient="records")
display(conflicts_df.head())

# Convert list values (e.g., spans, forms) to tuples so they are hashable
list_like_cols = [
    col
    for col in conflicts_df.columns
    if conflicts_df[col].apply(lambda x: isinstance(x, list)).any()
]
for col in list_like_cols:
    conflicts_df[col] = conflicts_df[col].apply(
        lambda x: tuple(x) if isinstance(x, list) else x
    )

# Check duplicates based on the row content
duplicates = conflicts_df.duplicated(keep=False)
duplicate_rows = conflicts_df[duplicates]

# Sort duplicate rows for better readability
duplicate_rows = duplicate_rows.sort_values(by=conflicts_df.columns.tolist())
print(f"Number of duplicate rows: {duplicate_rows.shape[0]}")
print("Duplicate rows:")
display(duplicate_rows)

# Remove duplicates
conflicts_df_no_duplicates = conflicts_df.drop_duplicates()
print(f"Number of rows before removing duplicates: {conflicts_df.shape[0]}")
print(f"Number of rows after removing duplicates: {conflicts_df_no_duplicates.shape[0]}")

# Check NaN values in the dataset
nan_counts = conflicts_df_no_duplicates.isna().sum()
print("NaN values in each column:")
print(nan_counts)

,sentence_id,sentence,word_form,span,current_deprel,current_case,current_analysis,bert_morph_v2_form
0,12410790,"Kristina Shmiguni ahistas Holmenkollenis 30 km klassikatehnikasõidu lõpukilomeetritel kurnatus , ent viiendast kohast maailma karikasarja liidrirüü hoidmiseks piisas .",Kristina,"[0, 8]",nsubj,gen,"gen,prop,sg",[sg n]
1,16715193,"Küpsetamist-kaunistamist jätkub tegelikult aasta lõpuni : sel pühapäeval aitab Torupilli Selveris piparkooke teha ja valmis piparkoogimaja detaile kokku panna pagar , järgmisel pühapäeval on tal kaubamaja kodumaailmas abiks Pipi .",Torupilli,"[79, 88]",nsubj,gen,"gen,prop,sg",[sg g]
2,214160,"Kui 20. sajandi algul laenu- ja hoiuühistud aitasid vältida eestlaste hoiustevoolu Venemaale , siis 21. sajandi algul võiks neist abi olla hoiuste äravoolu takistamisel ääremaalt ettevõtluskeskustesse .",laenu-,"[22, 28]",nsubj,gen,"com,gen,sg",[sg g]
3,225510,Tarkvara aitab jagu saada mitmes ametkonnas võidutsevast liigsest bürokraatiast .,Tarkvara,"[0, 8]",nsubj,gen,"com,gen,sg",[sg n]
4,325158,"Praegu aitavad selle kala DNA-st saadud elektritaju- ja elektriimpulsiomadused ühendada tehiselundite sensoorsed ja motoorsed signaalid tehisnaha pinnaga , kust närvisiirded neid edasi toimetavad .",kala,"[21, 25]",nsubj,gen,"com,gen,sg",[sg g]


Number of duplicate rows: 0
Duplicate rows:


,sentence_id,sentence,word_form,span,current_deprel,current_case,current_analysis,bert_morph_v2_form


Number of rows before removing duplicates: 9599
Number of rows after removing duplicates: 9599
NaN values in each column:
sentence_id           0
sentence              0
word_form             0
span                  0
current_deprel        0
current_case          0
current_analysis      0
bert_morph_v2_form    0
dtype: int64


In [130]:
conflicts_df_no_duplicates["bert_morph_v2_form"].value_counts()

bert_morph_v2_form
(sg n,)      5185
(sg p,)      1626
(sg g,)      1596
(sg all,)     483
(,)           208
(sg tr,)      158
(pl p,)        97
(adt,)         84
(sg el,)       25
(?,)           25
(o,)           19
(sg ad,)       10
(ks,)           9
(pl g,)         8
(vat,)          8
(sid,)          8
(sg kom,)       7
(neg o,)        7
(sg ill,)       6
(pl all,)       4
(sg in,)        4
(sg ter,)       4
(sg es,)        3
(ge,)           3
(pl ad,)        3
(neg ks,)       1
(ma,)           1
(da,)           1
(te,)           1
(s,)            1
(pl es,)        1
(neg ge,)       1
(sg abl,)       1
(pl n,)         1
Name: count, dtype: int64

In [131]:
# Map the given forms from Vabamorf to UD format
map_forms = {
    # Counts
    "sg": "sg",
    "pl": "pl",
    # Countable forms
    "n": "nom",
    "g": "gen",
    "p": "par",
    "adt": "ill",
    "ill": "ill",
    "in": "ine",
    "el": "ela",
    "all": "all",
    "ad": "ade",
    "abl": "abl",
    "tr": "tra",
    "ter": "trm",
    "es": "ess",
    "ab": "abe",
    "kom": "com",
}

# Map the given forms from UD to Vabamorf format
ud_map_forms = {"el": "el", "all": "all", "gen": "g", "adit": "adt", "": "-", "-": "-"}

<table cellspacing="0" border="1">
<tbody>
<tr><th>GT, giellalt</th><td> </td><td> </td><th>FS</th><th>CG</th><th>UD</th></tr>
<tr><th colspan="6">arv ||| number</th></tr>
<tr><td>Sg</td><td> ainsus</td><td> singular</td><td>sg</td><td>sg</td><td>Number=Sing</td></tr>
<tr><td>Pl</td><td> mitmus</td><td> plural</td><td>pl</td><td>pl</td><td>Number=Plur</td></tr>

<tr><th colspan="6">kääne ||| case</th></tr>
<tr><td>Nom</td><td> nimetav</td><td>nominative</td><td>n</td><td>nom</td><td>Case=Nom</td></tr>
<tr><td>Gen</td><td> omastav </td><td>genitive</td><td>g</td><td>gen</td><td>Case=Gen</td></tr>
<tr><td>Par</td><td> osastav</td><td>partitive</td><td>p</td><td>part</td><td>Case=Par</td></tr>
<tr><td>Ill</td><td> sisseütlev </td><td>illative</td><td>ill</td><td>ill</td><td>Case=Ill</td></tr>
<tr><td></td><td> suunduv (lühike sisseütlev)</td><td>additive</td><td>adt</td><td>adit</td><td>Case=Add</td></tr>
<tr><td>Ine</td><td> seesütlev</td><td>inessive</td><td>in</td><td>in</td><td>Case=Ine</td></tr>
<tr><td>Ela</td><td> seestütlev</td><td>elative</td><td>el</td><td>el</td><td>Case=Ela</td></tr>
<tr><td>All</td><td> alaleütlev</td><td>allative</td><td>all</td><td>all</td><td>Case=All</td></tr>
<tr><td>Ade</td><td> alalütlev</td><td>adessive</td><td>ade</td><td>ad</td><td>Case=Ade</td></tr>
<tr><td>Abl</td><td> alaltütlev</td><td>ablative</td><td>abl</td><td>abl</td><td>Case=Abl</td></tr>
<tr><td>Tra</td><td> saav</td><td>translative</td><td>tr</td><td>tr</td><td>Case=Tra</td></tr>
<tr><td>Trm</td><td> rajav</td><td>terminative</td><td>ter</td><td>term</td><td>Case=Ter</td></tr>
<tr><td>Ess</td><td> olev</td><td>essive</td><td>es</td><td>es</td><td>Case=Ess</td></tr>
<tr><td>Abe</td><td> ilmaütlev</td><td>abessive</td><td>ab</td><td>abes</td><td>Case=Abe</td></tr>
<tr><td>Com</td><td> kaasaütlev</td><td>comitative</td><td>kom</td><td>kom</td><td>Case=Com</td></tr>
</tbody></table>

&emsp;

<table cellspacing="0" border="1">
<tbody>
<tr><th>GT, giellalt</th><th>näide </th><th>FS</th><th>CG</th><th>UD</th></tr>
<tr><td>Inf</td><td> elada</td><td>da</td><td> inf</td><td>VerbForm=Inf</td></tr>
<tr><td>Ger</td><td> elades</td><td>des</td><td> ger</td><td>VerbForm=Conv</td></tr>
<tr><td>Pers Sup Ill</td><td> elama </td><td>ma</td><td> sup ps ill</td><td>Voice=Act VerbForm=Sup Case=Ill</td></tr>
<tr><td>Pers Sup Ine</td><td> elamas</td><td>mas</td><td> sup ps in</td><td>Voice=Act VerbForm=Sup Case=Ine</td></tr>
<tr><td>Pers Sup Ela</td><td> elamast</td><td>mast</td><td> sup ps el</td><td>Voice=Act VerbForm=Sup Case=Ela</td></tr>
<tr><td>Pers Sup Tra</td><td> elamaks</td><td>maks</td><td> sup ps tr</td><td>Voice=Act VerbForm=Sup Case=Tra</td></tr>
<tr><td>Pers Sup Abe</td><td> elamata</td><td>mata</td><td> sup ps abes</td><td>Voice=Act VerbForm=Sup Case=Abe</td></tr>

<tr><td>Pers Prs Ind Sg1 Aff</td><td>elan </td><td>n</td><td> indic pres ps1 sg ps af</td><td>Voice=Act Tense=Pres Mood=Ind VerbForm=Fin Number=Sing Person=1</td></tr>
<tr><td>Pers Prs Ind Sg2 Aff</td><td>elad </td><td>d</td><td> indic pres ps2 sg ps af</td><td>Voice=Act Tense=Pres Mood=Ind VerbForm=Fin Number=Sing Person=2</td></tr>
<tr><td>Pers Prs Ind Sg3 Aff</td><td>elab </td><td>b</td><td> indic pres ps3 sg ps af</td><td>Voice=Act Tense=Pres Mood=Ind VerbForm=Fin Number=Sing Person=3</td></tr>
<tr><td>Pers Prs Ind Pl1 Aff</td><td>elame </td><td>me</td><td> indic pres ps1 pl ps af</td><td>Voice=Act Tense=Pres Mood=Ind VerbForm=Fin Number=Plur Person=1</td></tr>
<tr><td>Pers Prs Ind Pl2 Aff</td><td>elate </td><td>te</td><td> indic pres ps2 pl ps af</td><td>Voice=Act Tense=Pres Mood=Ind VerbForm=Fin Number=Plur Person=2</td></tr>
<tr><td>Pers Prs Ind Pl3 Aff</td><td>elavad</td><td>vad</td><td> indic pres ps3 pl ps af</td><td>Voice=Act Tense=Pres Mood=Ind VerbForm=Fin Number=Plur Person=3</td></tr>

<tr><td>Pers Prs Ind Neg</td><td> ela (1)</td><td>o</td><td> indic pres ps neg</td><td>Voice=Act Tense=Pres Mood=Ind VerbForm=Fin Connegative=Yes</td></tr>

<tr><td>Pers Prs Cond Sg1 Aff</td><td> elaksin</td><td>ksin</td><td> cond pres ps1 sg ps af</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Sing Person=1</td></tr>
<tr><td>Pers Prs Cond Sg2 Aff</td><td> elaksid (2)</td><td>ksid</td><td> cond pres ps2 sg ps af</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Sing Person=2</td></tr>
<tr><td>Pers Prs Cond</td><td>elaks </td><td>ks</td><td> cond pres ps[1, 2, 3] [sg, pl] ps af;<br>  cond pres ps neg</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin;<br>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Connegative=Yes</td></tr>

<tr><td>Pers Prs Cond Pl1 Aff</td><td> elaksime</td><td>ksime</td><td> cond pres ps1 pl ps af</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Plur Person=1</td></tr>
<tr><td>Pers Prs Cond Pl2 Aff</td><td> elaksite</td><td>ksite</td><td> cond pres ps2 pl ps af</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Plur Person=2</td></tr>
<tr><td>Pers Prs Cond Pl3 Aff</td><td> elaksid (2)</td><td>ksid</td><td> cond pres ps3 pl ps af</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Plur Person=3</td></tr>

<tr><td>Pers Prs Imprt Sg2</td><td> ela (1)</td><td>o</td><td> imper pres ps2 sg ps [af, neg]</td><td>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Number=Plur Person=2;<br>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Number=Plur Person=2 Connegative=Yes</td></tr>
<tr><td>Pers Prs Imprt</td><td> elagu </td><td>gu</td><td> imper pres ps3 [sg, pl] ps [af, neg]</td><td>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin;<br>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Connegative=Yes </td></tr>

<tr><td>Pers Prs Imprt Pl1</td><td> elagem</td><td>gem</td><td> imper pres ps1 pl ps [af, neg]</td><td>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Number=Plur Person=1;<br>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Number=Plur Person=1 Connegative=Yes</td></tr>
<tr><td>Pers Prs Imprt Pl2</td><td> elage </td><td>ge</td><td> imper pres ps2 pl ps [af, neg]</td><td>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Number=Plur Person=2;<br>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Number=Plur Person=2 Connegative=Yes</td></tr>

<tr><td>Pers Prs Quot</td><td> elavat</td><td>vat</td><td> quot pres ps [af, neg]</td><td>Voice=Act Tense=Pres Mood=Qot VerbForm=Fin;<br>Voice=Act Tense=Pres Mood=Qot VerbForm=Fin Connegative=Yes</td></tr>

<tr><td>Pers Prs Prc</td><td> elav</td><td>v</td><td> partic pres ps</td><td>Voice=Act Tense=Pres VerbForm=Part <strong> ADJ Degree=Pos Number=Sing Case=Nom </strong></td></tr>

<tr><td>Pers Prt Ind Sg1 Aff</td><td> elasin</td><td>sin</td><td> indic impf ps1 sg ps af</td><td>Voice=Act Tense=Past Mood=Ind VerbForm=Fin Number=Sing Person=1</td></tr>
<tr><td>Pers Prt Ind Sg2 Aff</td><td> elasid (3)</td><td>sid</td><td> indic impf ps2 sg ps af</td><td>Voice=Act Tense=Past Mood=Ind VerbForm=Fin Number=Sing Person=2</td></tr>
<tr><td>Pers Prt Ind Sg3 Aff</td><td> elas </td><td>s</td><td> indic impf ps3 sg ps af</td><td>Voice=Act Tense=Past Mood=Ind VerbForm=Fin Number=Sing Person=3</td></tr>
<tr><td>Pers Prt Ind Pl1 Aff</td><td> elasime</td><td>sime</td><td> indic impf ps1 pl ps af</td><td>Voice=Act Tense=Past Mood=Ind VerbForm=Fin Number=Plur Person=1</td></tr>
<tr><td>Pers Prt Ind Pl2 Aff</td><td> elasite</td><td>site</td><td> indic impf ps2 pl ps af</td><td>Voice=Act Tense=Past Mood=Ind VerbForm=Fin Number=Plur Person=2</td></tr>
<tr><td>Pers Prt Ind Pl3 Aff</td><td> elasid (3)</td><td>sid</td><td> indic impf ps3 pl ps af</td><td>Voice=Act Tense=Past Mood=Ind VerbForm=Fin Number=Plur Person=3</td></tr>

<tr><td>Pers Prt Ind Neg</td><td>elanud (4)</td><td>nud</td><td> indic impf ps neg</td><td>Voice=Act Tense=Past Mood=Ind VerbForm=Fin Connegative=Yes</td></tr>

<tr><td>Pers Prt Cond Sg1 Aff</td><td> elanuksin</td><td>nuksin</td><td> cond past ps1 sg ps af</td><td>Voice=Act Tense=Past Mood=Cnd VerbForm=Fin Number=Sing Person=1</td></tr>
<tr><td>Pers Prt Cond Sg2 Aff</td><td> elanuksid (5)</td><td>nuksid</td><td> cond past ps2 sg ps af</td><td>Voice=Act Tense=Past Mood=Cnd VerbForm=Fin Number=Sing Person=2</td></tr>
<tr><td>Pers Prt Cond</td><td> elanuks</td><td>nuks</td><td> cond past ps[1, 2, 3] [sg, pl] ps af;<br>  cond past ps neg</td><td>Voice=Act Tense=Past Mood=Cnd VerbForm=Fin;<br>Voice=Act Tense=Past Mood=Cnd VerbForm=Fin Connegative=Yes</td></tr>
<tr><td>Pers Prt Cond Pl1 Aff</td><td> elanuksime</td><td>nuksime</td><td> cond past ps1 pl ps af</td><td>Voice=Act Tense=Past Mood=Cnd VerbForm=Fin Number=Plur Person=1</td></tr>
<tr><td>Pers Prt Cond Pl2 Aff</td><td> elanuksite</td><td>nuksite</td><td> cond past ps2 pl ps af</td><td>Voice=Act Tense=Past Mood=Cnd VerbForm=Fin Number=Plur Person=2</td></tr>
<tr><td>Pers Prt Cond Pl3 Aff</td><td> elanuksid (5)</td><td>nuksid</td><td> cond past ps3 pl ps af</td><td>Voice=Act Tense=Past Mood=Cnd VerbForm=Fin Number=Plur Person=3</td></tr>

<tr><td>Pers Prt Imprt *</td><td>elanud (4)</td><td>--</td><td>--</td><td>--</td></tr>

<tr><td>Pers Prt Quot</td><td> elanuvat</td><td>nuvat</td><td> quot past ps [af, neg]</td><td>Voice=Act Tense=Past Mood=Qot VerbForm=Fin;<br>Voice=Act Tense=Past Mood=Qot VerbForm=Fin Connegative=Yes</td></tr>

<tr><td>Pers Prt Prc</td><td>elanud (4)</td><td>nud</td><td> partic past ps</td><td>Voice=Act Tense=Past VerbForm=Part</td></tr>

<tr><td>Impers Sup</td><td> elatama</td><td>tama</td><td> sup imps</td><td>Voice=Pass VerbForm=Sup</td></tr>
<tr><td>Impers Prs Ind Aff</td><td> elatakse</td><td>takse</td><td> indic pres imps af</td><td>Voice=Pass Tense=Pres Mood=Ind VerbForm=Fin</td></tr>
<tr><td>Impers Prs Ind Neg</td><td> elata</td><td>ta</td><td> indic pres imps neg</td><td>Voice=Pass Tense=Pres Mood=Ind VerbForm=Fin Connegative=Yes</td></tr>
 
<tr><td>Impers Prs Cond</td><td> elataks</td><td>taks</td><td> cond pres imps [af, neg]</td><td>Voice=Pass Tense=Pres Mood=Cnd VerbForm=Fin;<br>Voice=Pass Tense=Pres Mood=Cnd VerbForm=Fin Connegative=Yes</td></tr>

<tr><td>Impers Prs Imprt</td><td> elatagu</td><td>tagu</td><td> imper pres imps [af, neg]</td><td>Voice=Pass Tense=Pres Mood=Imp VerbForm=Fin;<br>Voice=Pass Tense=Pres Mood=Imp VerbForm=Fin Connegative=Yes</td></tr>

<tr><td>Impers Prs Quot</td><td> elatavat</td><td>tavat</td><td> quot pres imps [af, neg]</td><td>Voice=Pass Tense=Pres Mood=Qot VerbForm=Fin;<br>Voice=Pass Tense=Pres Mood=Qot VerbForm=Fin Connegative=Yes</td></tr>

<tr><td>Impers Prs Prc</td><td> elatav</td><td>tav</td><td> partic pres imps</td><td>Voice=Pass Tense=Pres VerbForm=Part <strong> ADJ Degree=Pos Number=Sing Case=Nom </strong></td></tr>
<tr><td>Impers Prt Ind Aff</td><td> elati </td><td>ti</td><td> indic impf imps af</td><td>Voice=Pass Tense=Past Mood=Ind VerbForm=Fin</td></tr>
<tr><td>Impers Prt Ind Neg</td><td> elatud (6)</td><td>tud</td><td> indic impf imps neg</td><td>Voice=Pass Tense=Past Mood=Ind VerbForm=Fin Connegative=Yes</td></tr>

<tr><td>Impers Prt Cond</td><td> elatuks</td><td>tuks</td><td> cond past imps [af, neg]</td><td>Voice=Pass Tense=Past Mood=Cnd VerbForm=Fin;<br>Voice=Pass Tense=Past Mood=Cnd VerbForm=Fin Connegative=Yes</td></tr>

<tr><td>Impers Prt Quot*</td><td> elatuvat</td><td>tuvat</td><td> quot past imps [af, neg]</td><td>--</td></tr>

<tr><td>Impers Prt Prc</td><td> elatud (6)</td><td>tud</td><td> partic past imps</td><td>Voice=Pass Tense=Past VerbForm=Part</td></tr>

<tr><th colspan="5">abiverbide <i>pole</i> (=<i>ei</i>+<i>ole</i>) ja <i>ära</i> erandlikud üksikvormid</th></tr>

<tr><td>Pers Prs Ind Neg</td><td> pole</td><td>neg o</td><td> indic pres ps neg</td><td>Voice=Act Tense=Pres Mood=Ind VerbForm=Fin Polarity=Neg</td></tr>

<tr><td>Pers Prs Cond Sg1 Neg</td><td> poleksin</td><td>--</td><td>--</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Sing Person=1 Polarity=Neg</td></tr>
<tr><td>Pers Prs Cond Sg2 Neg</td><td> poleksid (2)</td><td>--</td><td>--</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Sing Person=2 Polarity=Neg</td></tr>
<tr><td>Pers Prs Cond Neg</td><td>poleks</td><td>neg ks</td><td> cond pres ps neg</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Polarity=Neg</td></tr>
<tr><td>Pers Prs Cond Pl1 Neg</td><td> poleksime</td><td>--</td><td>--</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Plur Person=1 Polarity=Neg</td></tr>
<tr><td>Pers Prs Cond Pl2 Neg</td><td> poleksite</td><td>--</td><td>--</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Plur Person=2 Polarity=Neg</td></tr>
<tr><td>Pers Prs Cond Pl3 Neg</td><td> poleksid (2)</td><td>--</td><td>--</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Plur Person=3 Polarity=Neg</td></tr>

<tr><td>Pers Prs Imprt Sg2 Neg</td><td> ära</td><td>neg o</td><td> imper pres ps2 sg ps neg</td><td>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Number=Sing Person=2 Polarity=Neg</td></tr>
<tr><td>Pers Prs Imprt Neg</td><td> ärgu</td><td>neg gu</td><td> imper pres ps3 [sg, pl] ps neg</td><td>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Number=[Sing, Plur] Person=3 Polarity=Neg</td></tr>
<tr><td>Pers Prs Imprt Pl1 Neg</td><td> ärgem</td><td>neg gem</td><td> imper pres ps1 pl ps neg</td><td>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Number=Plur Person=1 Polarity=Neg</td></tr>
<tr><td>Pers Prs Imprt Pl1 Neg</td><td> ärme</td><td>neg me</td><td> imper pres ps1 pl ps neg</td><td>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Polarity=Neg</td></tr>
<tr><td>Pers Prs Imprt Pl2 Neg</td><td> ärge</td><td>neg ge</td><td> imper pres ps2 pl ps neg</td><td>Voice=Act Tense=Pres Mood=Imp VerbForm=Fin Number=Plur Person=2 Polarity=Neg</td></tr>

<tr><td>Pers Prs Quot Neg</td><td>polevat</td><td>neg vat</td><td> quot pres ps neg</td><td>Voice=Act Tense=Pres Mood=Qot VerbForm=Fin Polarity=Neg</td></tr>

<tr><td>Pers Prt Ind Neg</td><td>polnud </td><td>neg nud</td><td> indic impf ps neg</td><td>Voice=Act Tense=Past Mood=Ind VerbForm=Fin Polarity=Neg</td></tr>

<tr><td>Pers Prt Cond Sg1 Neg</td><td> polnuksin</td><td>--</td><td>--</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Sing Person=1 Polarity=Neg</td></tr>
<tr><td>Pers Prt Cond Sg2 Neg</td><td> polnuksid (5)</td><td>--</td><td>--</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Sing Person=2 Polarity=Neg</td></tr>
<tr><td>Pers Prt Cond Neg</td><td> polnuks</td><td>neg nuks</td><td> cond past ps neg</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Polarity=Neg</td></tr>
<tr><td>Pers Prt Cond Pl1 Neg</td><td> polnuksime</td><td>--</td><td>--</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Plur Person=1 Polarity=Neg</td></tr>
<tr><td>Pers Prt Cond Pl2 Neg</td><td> polnuksite</td><td>--</td><td>--</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Plur Person=2 Polarity=Neg</td></tr>
<tr><td>Pers Prt Cond Pl3 Neg</td><td> polnuksid (5)</td><td>--</td><td>--</td><td>Voice=Act Tense=Pres Mood=Cnd VerbForm=Fin Number=Plur Person=3 Polarity=Neg</td></tr>

<tr><td>Pers Prt Imprt Neg</td><td> ärnud</td><td>--</td><td>--</td><td>--</td></tr>
<tr><td>Pers Prt Quot Neg</td><td> polnuvat</td><td>--</td><td> quot past ps neg</td><td>Voice=Act Tense=Past Mood=Qot VerbForm=Fin Polarity=Neg</td></tr>

<tr><td>Impers Prs Ind Neg</td><td> polda </td><td>neg da</td><td> indic pres imps neg</td><td>Voice=Pass Tense=Pres Mood=Ind VerbForm=Fin Polarity=Yes</td></tr>
<tr><td>Impers Prs Cond Neg</td><td> poldaks</td><td>--</td><td>--</td><td>Voice=Pass Tense=Pres Mood=Cnd VerbForm=Fin Polarity=Neg</td></tr>
<tr><td>Impers Prs Imprt Neg</td><td> ärdagu</td><td>--</td><td>--</td><td>--</td></tr>
<tr><td>Impers Prs Quot Neg</td><td> poldavat</td><td>--</td><td>--</td><td>Voice=Pass Tense=Pres Mood=Qot VerbForm=Fin Polarity=Neg</td></tr>

<tr><td>Impers Prt Ind Neg</td><td> poldud </td><td>neg tud</td><td> indic impf imps neg</td><td>Voice=Pass Tense=Past VerbForm=Part Polarity=Neg</td></tr>
<tr><td>Impers Prt Cond Neg</td><td> polduks</td><td>--</td><td>--</td><td>Voice=Pass Tense=Past Mood=Cnd VerbForm=Fin Polarity=Neg</td></tr>

<tr><th colspan="5">erandlikud üksikvormid</th></tr>

<tr><td>Pers Prs </td><td> kuulukse,<br> tunnukse, näikse</td><td>NULL</td><td>af</td><td>Tense=Pres Mood=Ind VerbForm=Fin</td></tr>
<tr><td>Neg</td><td> ei </td><td>neg</td><td> neg</td><td>Polarity=Neg</td></tr>

</tbody></table>


In [132]:
def parse_bmt_form(form):
    # If the form is None or empty, return None
    if form is None or form == [""]:
        return "-", "-"
    form_data = form[0].split(" ")
    # If the form is related to a verb
    if len(form_data) == 1:
        if form_data is None or form_data == [""]:
            return "-", "-"
        word_form = form_data[0]
        return word_form, None
    # Split the form into count and form
    else:
        count = form_data[0]
        word_form = form_data[1]
        # Map the form to Vabamorf format
        mapped_form = map_forms.get(word_form, word_form)
        return word_form, mapped_form

In [133]:
# Collect rows to save multiple CSV files
# - one for all matching forms
# - one for all non-matching forms
# - one for all no predicted forms (bert morph v2 form is None or empty or -)
# - one for all non-matching forms where the Bert Morph v2 form is in ["n", "p"]
# - one for all non-matching forms where Bert Morph v2 form is other than ["n", "p"]
matching_forms = []
non_matching_forms = []
no_predicted_forms = []
non_matching_forms_with_np = []
non_matching_forms_without_np = []

unique_vabamorf_forms = set()

# Iterate through the DataFrame rows and compare the forms
for index, row in conflicts_df_no_duplicates.iterrows():
    sentence = row["sentence"]
    # Check if the sentence contains any newline and replace it with a space for better readability
    if "\n" in sentence:
        sentence = sentence.replace("\n", " ")
        row["sentence"] = sentence
    word = row["word_form"]
    word_span = row["span"]
    bert_Morph_v2_form = row["bert_morph_v2_form"]
    bmt_form, bmt_ud_parsed_form = parse_bmt_form(bert_Morph_v2_form)
    vabamorf_form = row["current_case"]
    vabamorf_vm_parsed_form = ud_map_forms.get(vabamorf_form, vabamorf_form)
    row["bert_morph_v2_form"] = bmt_form
    row["vabamorf_form"] = vabamorf_vm_parsed_form
    unique_vabamorf_forms.add(vabamorf_vm_parsed_form)
    if bmt_form is None:
        no_predicted_forms.append(row)
    if vabamorf_vm_parsed_form == bmt_form:
        matching_forms.append(row)
    else:
        non_matching_forms.append(row)
        if bmt_form in ["n", "p"]:
            non_matching_forms_with_np.append(row)
        else:
            non_matching_forms_without_np.append(row)

In [134]:
output_columns = [
    "sentence",
    "word_form",
    "span",
    "bert_morph_v2_form",
    "vabamorf_form",
    "sentence_id",
    "current_deprel",
    "current_analysis",
]

# Save the data to CSV files
matching_forms_df = pd.DataFrame(matching_forms, columns=output_columns)
non_matching_forms_df = pd.DataFrame(non_matching_forms, columns=output_columns)
no_predicted_forms_df = pd.DataFrame(no_predicted_forms, columns=output_columns)
non_matching_forms_with_np_df = pd.DataFrame(
    non_matching_forms_with_np, columns=output_columns
)
non_matching_forms_without_np_df = pd.DataFrame(
    non_matching_forms_without_np, columns=output_columns
)

output_dir = conflicts_dir / "processed" / "csv_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# if not output_dir.exists():
matching_forms_df.to_csv(output_dir / "matching_forms.csv", index=False)
non_matching_forms_df.to_csv(output_dir / "non_matching_forms.csv", index=False)
no_predicted_forms_df.to_csv(output_dir / "no_predicted_forms.csv", index=False)
non_matching_forms_with_np_df.to_csv(
    output_dir / "non_matching_forms_with_np.csv", index=False
)
non_matching_forms_without_np_df.to_csv(
    output_dir / "non_matching_forms_without_np.csv", index=False
)

In [135]:
# Count:
# - the number of matching forms
# - the number of non-matching forms
# - the number of no predicted forms (bert morph v2 form is None)
# - the number of non-matching forms where the Bert Morph v2 form is in ["n", "p"]

overall_count = conflicts_df_no_duplicates.shape[0]
matching_forms_count = len(matching_forms)
non_matching_forms_count = len(non_matching_forms)
no_predicted_forms_count = len(no_predicted_forms)
non_matching_forms_with_np_count = len(non_matching_forms_with_np)
non_matching_forms_without_np_count = len(non_matching_forms_without_np)
non_matching_forms_with_g_count = non_matching_forms_df[
    non_matching_forms_df["bert_morph_v2_form"].isin(["g"])
].shape[0]
non_matching_forms_value_counts = non_matching_forms_df[
    "bert_morph_v2_form"
].value_counts()

In [136]:
print(f"Unique Vabamorf forms in the dataset: {unique_vabamorf_forms}")

Unique Vabamorf forms in the dataset: {'el', 'all', '-', 'g', 'adt'}


In [138]:
print(
    f"Total forms count: {overall_count} (100.00%) (Percentages are calculated based on the total number of forms in the dataset)"
)
print(
    f"Matching forms count: {matching_forms_count} ({matching_forms_count / overall_count * 100:.2f}%)"
)
print(
    f"Non-matching forms count: {non_matching_forms_count} ({non_matching_forms_count / overall_count * 100:.2f}%)"
)
print(
    f"No predicted forms count: {no_predicted_forms_count} ({no_predicted_forms_count / overall_count * 100:.2f}%)"
)
print(
    f"Non-matching forms where Bert Morph v2 form is in ['n', 'p'] count: {non_matching_forms_with_np_count} ({non_matching_forms_with_np_count / overall_count * 100:.2f}%)"
)
print(
    f"Non-matching forms where Bert Morph v2 form is 'g' count: {non_matching_forms_with_g_count} ({non_matching_forms_with_g_count / overall_count * 100:.2f}%)"
)
print("All non-matching forms counts:")
sorted_non_matching_forms = non_matching_forms_value_counts.sort_values(ascending=False)
for form, count in sorted_non_matching_forms.items():
    print(f"  {form}: {count} ({count / overall_count * 100:.2f}%)")

Total forms count: 9599 (100.00%) (Percentages are calculated based on the total number of forms in the dataset)
Matching forms count: 1717 (17.89%)
Non-matching forms count: 7882 (82.11%)
No predicted forms count: 0 (0.00%)
Non-matching forms where Bert Morph v2 form is in ['n', 'p'] count: 6909 (71.98%)
Non-matching forms where Bert Morph v2 form is 'g' count: 63 (0.66%)
All non-matching forms counts:
  n: 5186 (54.03%)
  p: 1723 (17.95%)
  all: 487 (5.07%)
  tr: 158 (1.65%)
  -: 76 (0.79%)
  adt: 65 (0.68%)
  g: 63 (0.66%)
  o: 26 (0.27%)
  ?: 25 (0.26%)
  ad: 13 (0.14%)
  ks: 10 (0.10%)
  sid: 8 (0.08%)
  vat: 8 (0.08%)
  kom: 7 (0.07%)
  ill: 6 (0.06%)
  es: 4 (0.04%)
  in: 4 (0.04%)
  ge: 4 (0.04%)
  ter: 4 (0.04%)
  ma: 1 (0.01%)
  da: 1 (0.01%)
  te: 1 (0.01%)
  s: 1 (0.01%)
  abl: 1 (0.01%)


In [122]:
printed = 0
for i, row in tqdm.tqdm(enumerate(conflicts_data), total=len(conflicts_data)):
    sentence = row["sentence"]
    word = row["word_form"]
    word_span = row["span"]
    bert_Morph_v2_form = row["bert_morph_v2_form"]
    bmt_form, parsed_form = parse_bmt_form(bert_Morph_v2_form)
    vabamorf_form = row["current_case"]
    # print(f"Sentence: {sentence}")
    # print(f"Word: {word}, Span: {word_span}")
    # print(f"Bert Morph v2 form: {bert_Morph_v2_form}")
    # print(f"Vabamorf form: {vabamorf_form}")
    # if bmt_form not in ["n", "p"] and parsed_form != vabamorf_form:
    # if bmt_form in ["g"] and vabamorf_form != parsed_form:
    if vabamorf_form in ["el"]:
        print(f"Diff {i}:")
        print(f" Sentence: {sentence}")
        print(f"  Word: {word}, Span: {word_span}")
        print(
            f"  Bert Morph v2 form: {bert_Morph_v2_form}, Parsed form: {parsed_form}, Mapped form: {bmt_form}"
        )
        print(f"  Vabamorf form: {vabamorf_form}")
        printed += 1
    if printed >= 25:
        break

 62%|██████▏   | 5974/9634 [00:00<00:00, 944576.17it/s]

Diff 569:
 Sentence: Kuuest Kanada hokivaatlejast viis ennustasid , et Bertuzzi teeb hea hooaja .
  Word: Kuuest, Span: [0, 6]
  Bert Morph v2 form: ['sg el'], Parsed form: ela, Mapped form: el
  Vabamorf form: el
Diff 1695:
 Sentence: Ligi neli viiest ameeriklasest , sealhulgas 70 protsenti vabariiklastest kardab , et Iraagis puhkeb kodusõda .
  Word: viiest, Span: [10, 16]
  Bert Morph v2 form: ['sg el'], Parsed form: ela, Mapped form: el
  Vabamorf form: el
Diff 3494:
 Sentence: Selle rünnakuga , mille viiest ÜRO Julgeolekunõukogu alalisest liikmest kolm hukka mõistsid , näitasid maailma vägevad , mida nad maailmaorganisatsioonist arvavad .
  Word: viiest, Span: [24, 30]
  Bert Morph v2 form: ['sg el'], Parsed form: ela, Mapped form: el
  Vabamorf form: el
Diff 4878:
 Sentence: Pereisa , 32 , nentis , et miljonist piisab tänases Eestis tegelikult väga väheseks .
  Word: miljonist, Span: [27, 36]
  Bert Morph v2 form: ['sg el'], Parsed form: ela, Mapped form: el
  Vabamorf form: el
D

In [ ]:
from estnltk_neural.taggers import StanzaSyntaxTagger

stanza_syntax_tagger = StanzaSyntaxTagger(
    input_type="morph_analysis",
    input_morph_layer="morph_analysis",
    add_parent_and_children=True,
)
text_obj = estnltk.Text(
    "Mis ma oskan sul kosta , proovi jah uuemaid draivereid , proovi mälu CAS 2 kui ei aita , siis oled stuck ..."
)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)
display(text_obj.stanza_syntax)
display(text_obj.morph_extended)

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc', 'parent_span', 'children'), spans=SL[Span('Mis', [{'id': 1, 'lemma': 'mis', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('pl', 'pl'), ('n', 'n')]), 'head': 3, 'deprel': 'obj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('ma', [{'id': 2, 'lemma': 'mina', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('sg', 'sg'), ('n', 'n')]), 'head': 3, 'deprel': 'nsubj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('oskan', [{'id': 3, 'lemma': 'oskama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('n', 'n')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_', 'parent_span': None, 'children': <class 'tuple'>}]),
Span('sul', [{'id': 4, 'lemma': 'sina', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('sg', 'sg'), ('ad', 'ad')]), 'head': 3, 'deprel': 'nsubj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('kosta', [{'id': 5, 'lemma': 'kostma', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('da', 'da')]), 'head': 3, 'deprel': 'conj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span(',', [{'id': 6, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 7, 'deprel': 'punct', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('proovi', [{'id': 7, 'lemma': 'proovima', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('o', 'o')]), 'head': 3, 'deprel': 'conj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('jah', [{'id': 8, 'lemma': 'jah', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 7, 'deprel': 'advmod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('uuemaid', [{'id': 9, 'lemma': 'uuem', 'upostag': 'C', 'xpostag': 'C', 'feats': OrderedDict([('pl', 'pl'), ('p', 'p')]), 'head': 10, 'deprel': 'amod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('draivereid', [{'id': 10, 'lemma': 'draiver', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('pl', 'pl'), ('p', 'p')]), 'head': 7, 'deprel': 'obj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span(',', [{'id': 11, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 12, 'deprel': 'punct', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('proovi', [{'id': 12, 'lemma': 'proov', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('sg', 'sg'), ('g', 'g')]), 'head': 7, 'deprel': 'conj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('mälu', [{'id': 13, 'lemma': 'mälu', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('sg', 'sg'), ('g', 'g')]), 'head': 12, 'deprel': 'obj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('CAS', [{'id': 14, 'lemma': 'CAS', 'upostag': 'Y', 'xpostag': 'Y', 'feats': OrderedDict([('?', '?')]), 'head': 13, 'deprel': 'appos', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('2', [{'id': 15, 'lemma': '2', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('?', '?')]), 'head': 14, 'deprel': 'flat', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('kui', [{'id': 16, 'lemma': 'kui', 'upostag': 'J', 'xpostag': 'J', 'feats': OrderedDict(), 'head': 18, 'deprel': 'mark', 'deps': '_', 'misc': '

Layer(name='morph_extended', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech', 'punctuation_type', 'pronoun_type', 'letter_case', 'fin', 'verb_extension_suffix', 'subcat'), spans=SL[Span('Mis', [{'normalized_text': 'Mis', 'lemma': 'mis', 'root': 'mis', 'root_tokens': ['mis'], 'ending': '0', 'clitic': '', 'form': 'sg nom', 'partofspeech': 'P', 'punctuation_type': None, 'pronoun_type': ['inter_rel'], 'letter_case': 'cap', 'fin': None, 'verb_extension_suffix': [], 'subcat': None}, {'normalized_text': 'Mis', 'lemma': 'mis', 'root': 'mis', 'root_tokens': ['mis'], 'ending': '0', 'clitic': '', 'form': 'pl nom', 'partofspeech': 'P', 'punctuation_type': None, 'pronoun_type': ['inter_rel'], 'letter_case': 'cap', 'fin': None, 'verb_extension_suffix': [], 'subcat': None}]),
Span('ma', [{'normalized_text': 'ma', 'lemma': 'mina', 'root': 'mina', 'root_tokens': ['mina'], 'ending': '0', 'clitic': '', 'form': 'sg nom', 'partofspeech': 'P', 'punctuation_type': None, 'pronoun_type': ['ps1'], 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': None}]),
Span('oskan', [{'normalized_text': 'oskan', 'lemma': 'oskama', 'root': 'oska', 'root_tokens': ['oska'], 'ending': 'n', 'clitic': '', 'form': 'mod indic pres ps1 sg ps af', 'partofspeech': 'V', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': True, 'verb_extension_suffix': [], 'subcat': ['Part', 'InfP']}, {'normalized_text': 'oskan', 'lemma': 'oskama', 'root': 'oska', 'root_tokens': ['oska'], 'ending': 'n', 'clitic': '', 'form': 'aux indic pres ps1 sg ps af', 'partofspeech': 'V', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': True, 'verb_extension_suffix': [], 'subcat': ['Part', 'InfP']}, {'normalized_text': 'oskan', 'lemma': 'oskama', 'root': 'oska', 'root_tokens': ['oska'], 'ending': 'n', 'clitic': '', 'form': 'main indic pres ps1 sg ps af', 'partofspeech': 'V', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': True, 'verb_extension_suffix': [], 'subcat': ['Part', 'InfP']}]),
Span('sul', [{'normalized_text': 'sul', 'lemma': 'sina', 'root': 'sina', 'root_tokens': ['sina'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'P', 'punctuation_type': None, 'pronoun_type': ['ps2'], 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': None}]),
Span('kosta', [{'normalized_text': 'kosta', 'lemma': 'kostma', 'root': 'kost', 'root_tokens': ['kost'], 'ending': 'a', 'clitic': '', 'form': 'mod inf', 'partofspeech': 'V', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': False, 'verb_extension_suffix': [], 'subcat': ['Intr']}, {'normalized_text': 'kosta', 'lemma': 'kostma', 'root': 'kost', 'root_tokens': ['kost'], 'ending': 'a', 'clitic': '', 'form': 'aux inf', 'partofspeech': 'V', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': False, 'verb_extension_suffix': [], 'subcat': ['Intr']}, {'normalized_text': 'kosta', 'lemma': 'kostma', 'root': 'kost', 'root_tokens': ['kost'], 'ending': 'a', 'clitic': '', 'form': 'main inf', 'partofspeech': 'V', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': False, 'verb_extension_suffix': [], 'subcat': ['Intr']}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z', 'punctuation_type': 'Com', 'pronoun_type': None, 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': None}]),
Span('proovi', [{'normalized_text': 'proovi', 'lemma': 'proovima', 'root': 'proovi', 'root_tokens': ['proovi'], 'ending': '0', 'clitic': '', 'form': 'mod indic pres ps neg', 'partofspeech': 'V', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': True, 'verb_extension_suffix': [], 'subcat': ['Part-P', 'InfP']}, {'normalized_text': 'proovi', 'lemma': 'proovima', 'root': 'proovi', 'root_tokens': ['proovi'], 'e

In [ ]:
print(dir(text_obj.stanza_syntax))

# Output the word "proovi"
display(text_obj.stanza_syntax[11])
display(text_obj.morph_extended[11])
display(text_obj.morph_analysis[11])

['TRANSLUCENT_NONE_VALUES', '__class__', '__copy__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_repr_html_', '_span_list', 'add_annotation', 'add_span', 'ambiguous', 'ancestor_layers', 'attribute_values', 'attributes', 'check_span_consistency', 'clear_spans', 'count_values', 'default_values', 'descendant_layers', 'diff', 'display', 'enclosing_text', 'end', 'enveloping', 'get', 'get_overview_dataframe', 'groupby', 'index', 'layer', 'meta', 'name', 'parent', 'print_start_end', 'remove_span', 'resolve_attribute', 'rolling', 'secondary_attributes', 'serialisation_module', 'span_l

text,id,lemma,upostag,xpostag,feats,head,deprel,deps,misc,parent_span,children
proovi,12,proov,S,S,"OrderedDict([('sg', 'sg'), ('g', 'g')])",7,conj,_,_,"Span('proovi', [{'id': 7, 'lemma': 'proovima', 'upostag': 'V', 'xpostag': 'V', ' ..., type: <class 'estnltk_core.layer.span.Span'>","(""Span(',', [{'id': 11, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': O ..., type: <class 'tuple'>, length: 3"


text,normalized_text,lemma,root,root_tokens,ending,clitic,form,partofspeech,punctuation_type,pronoun_type,letter_case,fin,verb_extension_suffix,subcat
proovi,proovi,proov,proov,['proov'],0,,com sg gen,S,None,None,None,None,[],None


text,normalized_text,lemma,root,root_tokens,ending,clitic,form,partofspeech
proovi,proovi,proov,proov,['proov'],0,,sg g,S
